# Assignment 4 - Intensity transformations

> 1. WAP to enhance the input image using **Log transform**. Assume c=1.
> 2. WAP to enhance the input image using **Power law / gamma transform**. Assume c=1,
>    and input the value of gamma from the user.
> 3. WAP to enhance the input image using **Intensity level slicing (contrast stretching)**.
>    The input range of intensity values and the desired range in the output image are
>    obtained from the user. (Example: input range 80 to 120 is stretched to 50-150.)

### What you actually learn here
- every one of these is a function `s = T(r)` applied **independently to each pixel**
- so all three are really just a 256-entry lookup table, which is how OpenCV does it
- how to read a transfer curve and predict the effect on the histogram
- the normalisation subtlety hiding in "assume c=1"

In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

dark = cv2.imread("images/board.jpg", cv2.IMREAD_GRAYSCALE)
lena = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
print("board mean:", round(dark.mean(), 1), " lena mean:", round(lena.mean(), 1))
_imgs, _titles = [dark, lena], ["board.jpg (dark, needs help)", "lena.jpg (well exposed)"]
fig, axes = plt.subplots(1, 2, figsize=(4 * 2, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3))
plt.hist(dark.ravel(), bins=256, range=(0, 256), color="steelblue")
plt.title("board.jpg my_calcHist - everything is crammed into the low end"); plt.xlabel("intensity"); plt.ylabel("pixel count")
plt.xlim(0, 255); plt.tight_layout(); plt.show()

## The mental model: a transfer curve

A point transform ignores neighbours completely. Pixel value in, pixel value out.
So the *entire* behaviour of the operation is one curve from `r` in `[0,255]` to `s` in `[0,255]`.
Plot the curve and you know what the image will do: curve above the diagonal = brighter,
below = darker, steeper than the diagonal = more contrast in that band (and necessarily less
somewhere else, since you only have 256 levels to spend).

Use this helper constantly - the curve is the thing you are designing.

In [ ]:
def plot_curve(*fns, labels=None, title="transfer curves"):
    r = np.tile(np.arange(256, dtype=np.uint8), (8, 1))                                     # (8,256) test image, row = 0..255
    plt.figure(figsize=(5, 5))
    plt.plot([0, 255], [0, 255], "k--", lw=1, label="identity")
    for i, f in enumerate(fns):
        plt.plot(np.arange(256), np.asarray(f(r))[0], lw=2,
                 label=labels[i] if labels else f"fn{i}")
    plt.xlabel("input intensity r"); plt.ylabel("output intensity s")
    plt.title(title); plt.legend(); plt.grid(alpha=.3); plt.xlim(0,255); plt.ylim(0,255)
    plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

plot_curve(lambda x: x, labels=["do nothing"], title="the identity, for reference")

## About "assume c=1"

The textbook log transform is

$$s = c \cdot \log(1 + r)$$

With `c = 1` and `r` up to 255, the largest output is `log(256) = 5.55`. As an 8-bit image
that is pure black - useless. So *every* practical implementation, and the one your tests
expect, keeps `c = 1` for the **shape** of the curve and then scales the result so the
output fills `[0, 255]`. That scaling is equivalent to choosing

$$c = \frac{255}{\log(1 + 255)}$$

Same for gamma: `s = c * r^gamma` with `c=1` only behaves if you work in normalised
`r/255` units and multiply back up by 255 at the end. Both facts are worth stating in your
write-up, because "assume c=1" is the part students get marked down on.

Useful numpy: `np.log1p(x)` computes `log(1+x)` accurately.

In [ ]:
r = np.arange(256, dtype=np.float64)
print("raw log(1+r) range :", np.log1p(r).min().round(3), "to", np.log1p(r).max().round(3))
print("as uint8           :", np.log1p(r).astype(np.uint8)[[0, 128, 255]], " <- black. useless.")
print("scaled to 0..255   :", (np.log1p(r) / np.log(256) * 255).round().astype(np.uint8)[[0, 128, 255]])

---
## Task 1 - log transform

Pin the endpoints: `0 -> 0` and `255 -> 255`. The curve must be non-decreasing and concave
(that is what "compresses the highlights, expands the shadows" means numerically).

In [ ]:
def my_log_transform(gray):
    """Log transform, s = c*log(1+r), scaled so the output spans the full 0..255 range.

    Parameters
    ----------
    gray : (H, W) uint8 array

    Returns
    -------
    (H, W) uint8 array. Monotonic non-decreasing in the input, with 0 -> 0 and 255 -> 255.
    Dark tones are lifted much more than bright ones.
    """
    # TODO: work in float, apply log(1+r), scale to 0..255, round, cast back to uint8
    raise NotImplementedError

### Test `my_log_transform` yourself

Fixture: the **ramp**: an `(8, 256)` uint8 image whose every row is `0, 1, ..., 255` (mean exactly 127.5). Look at one output row.

- uint8, same shape
- 0 → 0, and 255 → 255 (within 1): the output fills the range
- the row never decreases
- dark tones are lifted: input 64 comes out above 84
- the curve is concave: the second difference (`np.diff(row, 2)`) is never above 1
  (1, not 0, because uint8 rounding jitters it)
- on `board.jpg` the mean intensity goes up

In [ ]:
# Your tests for my_log_transform: one assert per property above.


In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# No cv2.logTransform exists. The idiom is: build a 256-entry LUT, apply cv2.LUT.
# cv2.LUT(src, lut) -- src must be uint8, lut must be a 256-element uint8 array.
import cv2, numpy as np
lut = np.round(np.log1p(np.arange(256)) / np.log1p(255) * 255).astype(np.uint8)
assert np.array_equal(cv2.LUT(dark, lut), my_log_transform(dark)), "disagrees with the LUT route"
print("match. cv2.LUT is how every point transform is done in production.")
print("Also useful: cv2.normalize(src, None, 0, 255, cv2.NORM_MINMAX) to rescale a float result.")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Four steps, one line each if you like:
1. `f = gray.astype(np.float64)`
2. `f = np.log1p(f)`
3. divide by the maximum the curve can reach - which is `np.log1p(255)` - and multiply by 255
4. `.round().astype(np.uint8)`

Step 3 is where the `c` from the textbook actually lives.

</details>

In [ ]:
plot_curve(my_log_transform, labels=["log"], title="log transform")
_imgs, _titles = [dark, my_log_transform(dark)], [f"input (mean {dark.mean():.0f})",
                                        f"log (mean {my_log_transform(dark).mean():.0f})"]
fig, axes = plt.subplots(1, 2, figsize=(4 * 2, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3))
plt.hist(dark.ravel(), bins=256, range=(0, 256), color="steelblue")
plt.title("before"); plt.xlabel("intensity"); plt.ylabel("pixel count")
plt.xlim(0, 255); plt.tight_layout(); plt.show()

---
## Task 2 - power law / gamma

$$s = 255 \cdot (r/255)^{\gamma}$$

`gamma = 1` must be the identity. `gamma < 1` brightens, `gamma > 1` darkens. Endpoints pinned
at 0 and 255 for every gamma. Your tests also compare the result against a lookup-table
implementation, so exact rounding matters.

In [ ]:
def my_gamma_LUT(gray, gamma):
    """Power-law (gamma) transform with c = 1 in normalised units.

    Parameters
    ----------
    gray  : (H, W) uint8 array
    gamma : float > 0. gamma < 1 brightens, gamma > 1 darkens, gamma == 1 is the identity.

    Returns
    -------
    (H, W) uint8 array, monotonic non-decreasing, with 0 -> 0 and 255 -> 255.
    """
    # TODO: normalise to 0..1, raise to the power, scale back, round, cast
    raise NotImplementedError

### Test `my_gamma_LUT` yourself

Fixture: the **ramp**: an `(8, 256)` uint8 image whose every row is `0, 1, ..., 255` (mean exactly 127.5).

- uint8, same shape; `gamma=1` is the identity (within 1)
- for each gamma in `0.4, 0.7, 1.0, 1.8, 2.5`: 0 → 0, 255 → 255 (within 1), and the row never decreases
- `gamma=0.4` maps 128 above 148; `gamma=2.5` maps 128 below 108
- at intensity 128, `gamma=0.5` is brighter than `gamma=1.5`
- on lena at `gamma=2.2`: agrees (within 2) with a second, independent implementation:
  build the 256-entry table yourself and apply it with `cv2.LUT`

In [ ]:
# Your tests for my_gamma_LUT: one assert per property above.


In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
import cv2, numpy as np
gamma = 2.2
lut = np.round((np.arange(256) / 255.0) ** gamma * 255).astype(np.uint8)
assert np.array_equal(cv2.LUT(lena, lut), my_gamma_LUT(lena, gamma)), "disagrees with the LUT route"
print("match. cv2.LUT(src, lut) with a gamma table IS how display gamma is applied.")
print("Related: cv2.pow(src, power) works on floats; cv2.convertScaleAbs(src, alpha, beta)")
print("         does |alpha*src + beta| and casts to uint8 in one call.")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Divide by 255.0 *before* the exponent - that is what makes `c=1` work and what pins
both endpoints (since 0^g = 0 and 1^g = 1). Then multiply by 255 and round.

If `gamma=1` is off by more than 1 somewhere, you truncated instead of rounding.

</details>

In [ ]:
gs = [0.3, 0.5, 1.0, 1.8, 3.0]
plot_curve(*[(lambda g: (lambda x: my_gamma_LUT(x, g)))(g) for g in gs],
           labels=[f"gamma={g}" for g in gs], title="power law")
_imgs, _titles = list([my_gamma_LUT(dark, g) for g in gs]), [f"gamma={g}" for g in gs]
_cols = 5; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
_imgs, _titles = list([my_gamma_LUT(lena, g) for g in gs]), [f"gamma={g}" for g in gs]
_cols = 5; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

### The lookup table, which is the point of point transforms

Only 256 possible inputs exist. Compute the answer 256 times, then index. `cv2.LUT` does the
indexing in C. This is not a micro-optimisation - it is how gamma correction is done everywhere,
including in your display's firmware.

In [ ]:
lut = ((np.arange(256) / 255.0) ** 2.2 * 255).round().astype(np.uint8)
big = cv2.resize(lena, (1920, 1080))
print("identical results:", np.array_equal(cv2.LUT(big, lut), my_gamma_LUT(big, 2.2)))
print("\ntiming on 1080p:")
%timeit -r3 -n5 my_gamma_LUT(big, 2.2)
%timeit -r3 -n5 cv2.LUT(big, lut)

---
## Task 3 - intensity level slicing / contrast stretching

Piecewise linear, **three** segments. With input range `[in_lo, in_hi]` mapped to
`[out_lo, out_hi]`:

```
  0      -> 0
  in_lo  -> out_lo          linear in between
  in_hi  -> out_hi          linear in between   <- this is the stretched band
  255    -> 255             linear in between
```

For the assignment's example (80,120) -> (50,150), the middle segment has slope
`(150-50)/(120-80) = 2.5`: detail inside that band gets 2.5x more levels, and the two
outer segments get squashed to pay for it.

Two edge cases you must test. `(0, 255, 0, 255)` must be the identity - so do not divide
by a zero-width segment. And `out_lo > out_hi` (say `(50,200) -> (200,50)`) must genuinely
invert that band - do not sort or clamp the arguments to "fix" it.

In [ ]:
def my_contrast_stretch(gray, in_lo, in_hi, out_lo, out_hi):
    """Three-segment piecewise-linear intensity level slicing.

    Maps 0->0, in_lo->out_lo, in_hi->out_hi, 255->255, linearly between those knots.

    Parameters
    ----------
    gray           : (H, W) uint8 array
    in_lo,  in_hi  : ints, the input band to stretch (in_lo < in_hi)
    out_lo, out_hi : ints, where that band lands in the output (may be inverted)

    Returns
    -------
    (H, W) uint8 array.
    """
    # TODO: your code here
    raise NotImplementedError

### Test `my_contrast_stretch` yourself

Fixture: the **ramp**: an `(8, 256)` uint8 image whose every row is `0, 1, ..., 255` (mean exactly 127.5), with `(in_lo, in_hi, out_lo, out_hi) = (80, 120, 50, 150)`.

- uint8, same shape
- 80 → 50 and 120 → 150 (within 1); the midpoint 100 → 100 (within 2)
- 0 → 0, 255 → 255 (within 1); the row never decreases
- the band is stretched: slope between 80 and 120 is above 1
- below `in_lo` the slope is gentler than inside the band
- `(0, 255, 0, 255)` is the identity (within 1)
- `(50, 200, 200, 50)` really inverts the band: output at 60 is brighter than output at 190

In [ ]:
# Your tests for my_contrast_stretch: one assert per property above.


In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
import cv2, numpy as np
lut = np.round(np.interp(np.arange(256), [0, 80, 120, 255], [0, 50, 150, 255])).astype(np.uint8)
assert np.array_equal(cv2.LUT(lena, lut), my_contrast_stretch(lena, 80, 120, 50, 150))
print("match.")
print("For the common 'stretch min..max to 0..255' case there is a dedicated call:")
print("  cv2.normalize(src, None, 0, 255, cv2.NORM_MINMAX)")
nrm = cv2.normalize(dark, None, 0, 255, cv2.NORM_MINMAX)
print(f"  board.jpg range {dark.min()}..{dark.max()}  ->  {nrm.min()}..{nrm.max()}")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

The lazy way and the instructive way:

**Lazy:** `np.interp` does piecewise-linear interpolation through a set of knots. You have exactly four knots. Feed it the x-list and the y-list, then round and cast. It handles the zero-width-segment case for free.

**Instructive:** build the 256-entry LUT with three explicit loops or three slice assignments, one per segment, then `cv2.LUT`. Do this one too - it forces you to write the line equation `out_lo + (r - in_lo) * slope` yourself.

</details>

In [ ]:
plot_curve(lambda x: my_contrast_stretch(x, 80, 120, 50, 150),
           lambda x: my_contrast_stretch(x, 100, 180, 0, 255),
           lambda x: my_contrast_stretch(x, 50, 200, 200, 50),
           labels=["(80,120)->(50,150)", "(100,180)->(0,255)", "inverted band"],
           title="intensity level slicing")
_imgs, _titles = [lena, my_contrast_stretch(lena, 80, 120, 50, 150), my_contrast_stretch(lena, 100, 180, 0, 255), my_contrast_stretch(lena, 50, 200, 200, 50)], ["input", "(80,120)->(50,150)", "(100,180)->(0,255)", "inverted band"]
fig, axes = plt.subplots(1, 4, figsize=(3.2 * 4, 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

### Choosing the band from the histogram instead of guessing

Real contrast stretching reads the input range off the histogram - typically the 2nd and 98th
percentile, so a handful of outlier pixels do not waste the whole range.

In [ ]:
lo, hi = np.percentile(dark, [2, 98])
print(f"board.jpg: 2nd pct = {lo:.0f}, 98th pct = {hi:.0f}  (full range 0-255 is mostly empty)")
auto = my_contrast_stretch(dark, int(lo), int(hi), 0, 255)
_imgs, _titles = [dark, auto, my_log_transform(dark), my_gamma_LUT(dark, 0.45)], ["input", f"auto stretch [{lo:.0f},{hi:.0f}]", "log", "gamma 0.45"]
fig, axes = plt.subplots(1, 4, figsize=(3.2 * 4, 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3))
plt.hist(dark.ravel(), bins=256, range=(0, 256), color="steelblue")
plt.title("before"); plt.xlabel("intensity"); plt.ylabel("pixel count")
plt.xlim(0, 255); plt.tight_layout(); plt.show()

---
## Think about it

1. All three transforms brighten `board.jpg`. Compare them on **detail in the bright regions**.
   Which one destroys the highlights, and why is that visible in its curve?
2. A stretch that maps `(80,120)->(50,150)` is lossy in a specific way. How many input levels
   map to the same output level in the `0..80` segment? Compute it, then explain what that
   does to a smooth gradient in a dark area (search "banding" if you need the word).
3. Is `my_gamma_LUT(gamma=0.5)` the same as `my_log_transform`? Overlay both curves with
   `plot_curve` and quantify the maximum difference.
4. Gamma with `gamma=2.2` is what a monitor does to the signal it receives. Given that, is
   an image file's stored value proportional to light intensity? What does that imply about
   averaging pixel values - as you did in assignment 02?

In [ ]:
# Scratch space.

---
## Stretch

1. **Histogram equalisation.** `cv2.equalizeHist(dark)` - compare against your auto-stretch.
   Then implement it yourself: the transfer curve *is* the normalised cumulative histogram,
   about 4 lines with `np.cumsum`. This is the natural next topic in the Gonzalez book.
2. **CLAHE.** `cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(dark)`. Explain what
   "adaptive" and "contrast limited" each fix, in one sentence apiece.
3. **Bit-plane slicing.** Show the 8 bit planes of `lena` (`(g >> k) & 1`). Which planes carry
   recognisable structure, and which are basically noise? Reconstruct the image from only the
   top 4 planes and report the mean absolute error.
4. **A general LUT engine.** Write `apply_lut(gray, fn)` that samples any Python `fn(r)->s` at
   the 256 integer inputs and applies it with `cv2.LUT`. Rewrite all three tasks as one-liners
   through it. That is the refactor the whole notebook has been pointing at.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - point transforms

```python
dst = cv2.LUT(src, lut)
```
`src` must be **uint8**; `lut` must have exactly **256** entries. This is how every
point transform is done in production — build the table in numpy (`int64`, so no overflow),
cast the table to `uint8` once, then let OpenCV do the indexing in C.

```python
dst = cv2.normalize(src, None, alpha, beta, norm_type)
```
With `norm_type=cv2.NORM_MINMAX`, rescales the actual min..max to `alpha..beta` — the
"stretch to full range" case of contrast stretching, in one call. Also `NORM_L1`, `NORM_L2`,
`NORM_INF`.

```python
dst = cv2.convertScaleAbs(src, alpha=1, beta=0)   # linear brightness/contrast, saturating
dst = cv2.pow(src, power)                         # float input
dst = cv2.log(src)  /  cv2.exp(src)               # float input, natural log
dst = cv2.equalizeHist(src)                       # notebook 09
```